# Excitement Analysis

Calculate actual game excitement from odds volatility during completed games.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm

from hype_score.database import get_engine, load_games, load_odds_snapshots
from hype_score.excitement import calculate_excitement_batch, calculate_excitement, get_excitement_level

## Load Completed Games with Odds Data

In [ ]:
engine = get_engine()

games = load_games(engine, status='final', limit=500)
print(f"Loaded {len(games):,} completed games")

In [ ]:
all_snapshots = load_odds_snapshots(engine, game_ids=games['id'].tolist())
print(f"Loaded {len(all_snapshots):,} odds snapshots")
print(f"Games with snapshots: {all_snapshots['game_id'].nunique()}")

## Calculate Excitement Scores

In [ ]:
excitement_results = calculate_excitement_batch(all_snapshots)
print(f"Calculated excitement for {len(excitement_results):,} games")
excitement_results.head(10)

In [ ]:
excitement_with_info = excitement_results.merge(
    games[['id', 'sport', 'home_team', 'away_team', 'home_score', 'away_score', 'scheduled_start']],
    left_on='game_id',
    right_on='id'
)
excitement_with_info.head()

## Excitement Distribution

In [ ]:
fig = px.histogram(
    excitement_with_info, 
    x='volatility_score', 
    color='sport',
    nbins=50,
    title='Excitement Score Distribution by Sport'
)
fig.add_vline(x=30, line_dash='dash', annotation_text='Low/Med threshold')
fig.add_vline(x=60, line_dash='dash', annotation_text='Med/High threshold')
fig.show()

In [ ]:
excitement_with_info.groupby('sport')[['volatility_score', 'finish_drama_score', 'lead_changes', 'max_comeback']].describe().round(1)

## Most Exciting Games

In [ ]:
most_exciting = excitement_with_info.nlargest(20, 'volatility_score')
most_exciting[['home_team', 'away_team', 'sport', 'home_score', 'away_score', 
               'volatility_score', 'lead_changes', 'max_comeback', 'scheduled_start']]

In [ ]:
top_game_id = most_exciting.iloc[0]['game_id']
top_game = most_exciting.iloc[0]

top_snapshots = all_snapshots[all_snapshots['game_id'] == top_game_id].sort_values('timestamp')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=top_snapshots['timestamp'],
    y=top_snapshots['home_win_probability'],
    mode='lines',
    name='Home Win Probability'
))
fig.add_hline(y=0.5, line_dash='dash', line_color='gray', annotation_text='50/50')
fig.update_layout(
    title=f"Most Exciting Game: {top_game['home_team']} vs {top_game['away_team']}<br>Score: {top_game['volatility_score']:.1f}",
    yaxis_title='Home Win Probability',
    yaxis_range=[0, 1]
)
fig.show()

## Least Exciting Games

In [ ]:
games_with_enough_data = excitement_with_info[excitement_with_info['snapshot_count'] >= 10]
least_exciting = games_with_enough_data.nsmallest(20, 'volatility_score')
least_exciting[['home_team', 'away_team', 'sport', 'home_score', 'away_score', 
                'volatility_score', 'lead_changes', 'snapshot_count', 'scheduled_start']]

## Excitement Components Correlation

In [ ]:
metrics = ['volatility_score', 'finish_drama_score', 'max_comeback', 'lead_changes', 'total_prob_movement']
print("Excitement metrics correlation:")
excitement_results[metrics].corr().round(2)

In [ ]:
fig = px.scatter(
    excitement_with_info,
    x='lead_changes',
    y='volatility_score',
    color='sport',
    hover_data=['home_team', 'away_team'],
    title='Lead Changes vs Volatility Score'
)
fig.show()